# Sales CSV - Bronze ingestion

**Medallion role:** Bronze. This notebook batch-loads product and inventory CSV files from ADLS into separate external Delta tables. It preserves the source-shaped records and adds lineage metadata; storage authentication is expected to be supplied by the workspace and Unity Catalog configuration rather than notebook secrets.

In [0]:
dbutils.widgets.removeAll()

## Runtime configuration and source-to-target mapping

Widgets select the environment and ingestion timestamp. Product reference data and inventory transactions have independent landing paths, tables, and external Delta paths because they have different business keys and update patterns.

In [0]:
from datetime import datetime, timezone

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

dbutils.widgets.text(
    "ingestion_timestamp",
    datetime.now(timezone.utc).isoformat(),
    "Ingestion Timestamp"
)

environment = dbutils.widgets.get("environment").lower()
ingestion_timestamp = (
    dbutils.widgets.get("ingestion_timestamp").strip()
)

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "storage_account": "stcentralusjrdev",
        "catalog": "salescsv_dev"
    },
    "prod": {
        "storage_account": "stcentralusjrprod",
        "catalog": "salescsv_prod"
    }
}

env = config[environment]

storage_account = env["storage_account"]
catalog = env["catalog"]

product_source_path = (
    f"abfss://landing@{storage_account}.dfs.core.windows.net/"
    "salescsv/product/"
)

inventory_source_path = (
    f"abfss://landing@{storage_account}.dfs.core.windows.net/"
    "salescsv/inventory/"
)

product_table = (
    f"{catalog}.bronze.product_catalog_raw"
)

inventory_table = (
    f"{catalog}.bronze.inventory_transactions_raw"
)

product_target_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salescsv/bronze/product_catalog_raw/"
)

inventory_target_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salescsv/bronze/inventory_transactions_raw/"
)

print("=" * 60)
print("SALES CSV - BRONZE INGESTION")
print("=" * 60)
print(f"Environment              : {environment}")
print(f"Ingestion timestamp      : {ingestion_timestamp}")
print(f"Catalog                  : {catalog}")
print(f"Product source           : {product_source_path}")
print(f"Inventory source         : {inventory_source_path}")
print(f"Product target           : {product_table}")
print(f"Inventory target         : {inventory_table}")
print("=" * 60)

## Batch CSV ingestion with inferred source types

Both datasets are read as finite batches with headers and `inferSchema` enabled. Bronze intentionally accepts the inferred source representation, while source file, file path, and ingestion timestamp columns provide the lineage required for auditing and later MERGE decisions.

In [0]:
from pyspark.sql.functions import (
    col,
    lit
)

product_raw_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(product_source_path)

        .withColumn(
            "source_file",
            col("_metadata.file_name")
        )

        .withColumn(
            "source_file_path",
            col("_metadata.file_path")
        )

        .withColumn(
            "ingestion_timestamp",
            lit(ingestion_timestamp).cast("timestamp")
        )
)


In [0]:
inventory_raw_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(inventory_source_path)

        .withColumn(
            "source_file",
            col("_metadata.file_name")
        )

        .withColumn(
            "source_file_path",
            col("_metadata.file_path")
        )

        .withColumn(
            "ingestion_timestamp",
            lit(ingestion_timestamp).cast("timestamp")
        )
)

## Idempotent external Delta persistence

The first load registers each dataset as an external Delta table at its configured path. Later runs MERGE on the dataset's business key, update a matched row only when the incoming ingestion timestamp is newer, and insert unseen keys, making repeated batch execution safe.

In [0]:
from delta.tables import DeltaTable


def merge_bronze_table(
    source_df,
    target_table,
    target_path,
    merge_key
):
    """
    Creates an external Bronze Delta table on the initial load.
    Subsequent executions perform an idempotent MERGE using the
    source business identifier.
    """

    if not spark.catalog.tableExists(target_table):

        print(
            f"Initial load. Creating Bronze table: "
            f"{target_table}"
        )

        (
            source_df.write
                .format("delta")
                .mode("append")
                .option("path", target_path)
                .saveAsTable(target_table)
        )

    else:

        print(
            f"Target exists. Merging into: "
            f"{target_table}"
        )

        target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target.alias("target")
                .merge(
                    source_df.alias("source"),
                    f"target.{merge_key} = source.{merge_key}"
                )

                .whenMatchedUpdateAll(
                    condition=(
                        "source.ingestion_timestamp "
                        "> target.ingestion_timestamp"
                    )
                )

                .whenNotMatchedInsertAll()

                .execute()
        )

    print(
        f"Bronze load completed: {target_table}"
    )

In [0]:
merge_bronze_table(
    source_df=product_raw_df,
    target_table=product_table,
    target_path=product_target_path,
    merge_key="product_id"
)

In [0]:
merge_bronze_table(
    source_df=inventory_raw_df,
    target_table=inventory_table,
    target_path=inventory_target_path,
    merge_key="transaction_id"
)